In [47]:
import sqlite3

conn = sqlite3.connect("E-commerces_database.db")
cursor = conn.cursor()
print("Connection and database created successfully")

cursor.execute("PRAGMA foreign_keys = ON")

cursor.execute("""
    CREATE TABLE IF NOT EXISTS Customers(
        Customer_ID INTEGER PRIMARY KEY,
        CustomerName TEXT NOT NULL,
        Email TEXT UNIQUE,
        City TEXT)
""")

conn.commit()
print("Customer table created successfully")

cursor.execute("""
    CREATE TABLE IF NOT EXISTS Products(
        Product_ID INTEGER PRIMARY KEY,
        ProductName TEXT NOT NULL,
        Category TEXT,
        UnitPrice REAL CHECK(UnitPrice > 0))
""")
conn.commit()
print("Products table created successfully")


cursor.execute("""
    CREATE TABLE IF NOT EXISTS Orders(
        OrderID INTEGER PRIMARY KEY,
        Customer_ID INTEGER,
        OrderDate DATE NOT NULL,
        OrderStatus TEXT CHECK(OrderStatus IN("Pending", "Shipped", "Delivered", "Cancelled")),

        FOREIGN  KEY(Customer_ID)
            REFERENCES Customers(Customer_ID)
            ON DELETE CASCADE
            ON UPDATE CASCADE
        
        )
""")
conn.commit()
print("Orders table created successfully")  


cursor.execute("""
    CREATE TABLE IF NOT EXISTS OrderItems(
        OrderID INTEGER,
        Product_ID INTEGER,
        Quantity INTEGER CHECK(Quantity > 0),
        UnitPrice REAL CHECK(UnitPrice > 0),
        PRIMARY KEY(OrderID, Product_ID),

        FOREIGN KEY (OrderID)
            REFERENCES Orders(OrderID)
            ON DELETE CASCADE
            ON UPDATE CASCADE,

        FOREIGN KEY ( Product_ID)
            REFERENCES Products(Product_ID)
            ON DELETE CASCADE
            ON UPDATE CASCADE)
""")

conn.commit()
print("OrderItems table created successfully")

cursor.execute("SELECT COUNT(*) FROM Customers")
count = cursor.fetchone()[0]

if count == 0:
    cursor.executemany("""
        INSERT INTO Customers
        (Customer_ID, CustomerName, Email, City)
        VALUES (?, ?, ?, ?)
        """, [
        (1, 'Anu', 'anu@gmail.com', 'Coimbatore'),
        (2, 'Ravi', 'ravi@gmail.com', 'Chennai'),
        (3, 'Priya', 'priya@gmail.com', 'Bangalore'),
        (4, 'Arun', 'arun@gmail.com', 'Hyderabad'),
        (5, 'Meena', 'meena@gmail.com', 'Pune')
        ])

    conn.commit()
    print("Customers data entered successfully")
else:
    print("Customer data already exists")
    
cursor.execute("SELECT COUNT(*) FROM Products")
count = cursor.fetchone()[0]

if count == 0:
    cursor.executemany("""
        INSERT INTO Products
        (Product_ID, ProductName, Category, UnitPrice)
        VALUES (?, ?, ?, ?)
        """, [
        (101, 'Laptop', 'Electronics', 55000),
        (102, 'Mouse', 'Accessories', 800),
        (103, 'Keyboard', 'Accessories', 1500),
        (104, 'Monitor', 'Electronics', 12000),
        (105, 'Headphones', 'Accessories', 2500)
        ])

    conn.commit()
    print("Products data entered successfully")
else:
    print("Products data already exists")

cursor.execute("SELECT COUNT(*) FROM Orders")
count = cursor.fetchone()[0]

if count == 0:
    cursor.executemany("""
        INSERT INTO Orders
        (OrderID, Customer_ID, OrderDate, OrderStatus)
        VALUES (?, ?, ?, ?)
        """, [
        (1001, 1, '2026-09-01', 'Delivered'),
        (1002, 2, '2026-09-03', 'Shipped'),
        (1003, 1, '2026-09-05', 'Pending'),
        (1004, 3, '2026-09-07', 'Delivered'),
        (1005, 4, '2026-09-10', 'Cancelled'),
        (1006, 5, '2026-09-12', 'Pending')
        ])

    conn.commit()
    print("Orders data entered successfully")
else:
    print("Orders data already exists")


cursor.execute("SELECT COUNT(*) FROM OrderItems")
count = cursor.fetchone()[0]

if count == 0:
    cursor.executemany("""
        INSERT INTO OrderItems
        (OrderID, Product_ID, Quantity, UnitPrice)
        VALUES (?, ?, ?, ?)
        """, [
        (1001, 101, 1, 55000),
        (1001, 102, 2, 800),
        (1002, 103, 1, 1500),
        (1002, 105, 2, 2500),
        (1003, 104, 1, 12000),
        (1004, 101, 1, 55000),
        (1004, 103, 1, 1500),
        (1005, 102, 3, 800),
        (1006, 105, 1, 2500),
        (1006, 104, 2, 12000)
        ])

    conn.commit()
    print("OrderItems data entered successfully")
else:
    print("OrderItems data already exists")

Connection and database created successfully
Customer table created successfully
Products table created successfully
Orders table created successfully
OrderItems table created successfully
Customers data entered successfully
Products data entered successfully
Orders data entered successfully
OrderItems data entered successfully


In [48]:
#Verifying foregin key on or not
cursor.execute("PRAGMA foreign_keys")
print(cursor.fetchall())

print()
cursor.execute("""
SELECT *
FROM Customers AS c
LEFT JOIN Orders AS o
ON c.Customer_ID = o.Customer_ID
""")

result = cursor.fetchall()
for r in result:
    print(r)

print()
cursor.execute("""
    INSERT INTO Customers
    (Customer_ID, CustomerName, Email, City)
    VALUES (1, 'Kumar', 'kumar@gmail.com', 'Chennai')
""")

[(1,)]

(1, 'Anu', 'anu@gmail.com', 'Coimbatore', 1001, 1, '2026-09-01', 'Delivered')
(1, 'Anu', 'anu@gmail.com', 'Coimbatore', 1003, 1, '2026-09-05', 'Pending')
(2, 'Ravi', 'ravi@gmail.com', 'Chennai', 1002, 2, '2026-09-03', 'Shipped')
(3, 'Priya', 'priya@gmail.com', 'Bangalore', 1004, 3, '2026-09-07', 'Delivered')
(4, 'Arun', 'arun@gmail.com', 'Hyderabad', 1005, 4, '2026-09-10', 'Cancelled')
(5, 'Meena', 'meena@gmail.com', 'Pune', 1006, 5, '2026-09-12', 'Pending')



IntegrityError: UNIQUE constraint failed: Customers.Customer_ID

In [49]:
#Composite foregin key
cursor.execute("""
INSERT INTO OrderItems
(OrderID, Product_ID, Quantity,UnitPrice)
VALUES (1001, 105, 2,8000)
""")

conn.commit()

cursor.execute("SELECT * FROM OrderItems")
result = cursor.fetchall()
for r in result:
    print(r)

cursor.execute("""
    INSERT INTO OrderItems
    (OrderID, Product_ID, Quantity, UnitPrice)
    VALUES (1001, 101, 2, 55000)
""")
#The pair (OrderID, Product_ID) is the primary key, not OrderID by itself.

(1001, 101, 1, 55000.0)
(1001, 102, 2, 800.0)
(1002, 103, 1, 1500.0)
(1002, 105, 2, 2500.0)
(1003, 104, 1, 12000.0)
(1004, 101, 1, 55000.0)
(1004, 103, 1, 1500.0)
(1005, 102, 3, 800.0)
(1006, 105, 1, 2500.0)
(1006, 104, 2, 12000.0)
(1001, 105, 2, 8000.0)


IntegrityError: UNIQUE constraint failed: OrderItems.OrderID, OrderItems.Product_ID

In [39]:
cursor.execute("""
    INSERT INTO Orders
    (OrderID, Customer_ID, OrderDate, OrderStatus)
    VALUES (2001, 999, '2026-09-23', 'Pending')
""")

IntegrityError: FOREIGN KEY constraint failed

In [40]:
cursor.execute("""
    INSERT INTO OrderItems
    (OrderID, Product_ID, Quantity, UnitPrice)
    VALUES (9999, 101, 1, 55000)
""")

IntegrityError: FOREIGN KEY constraint failed

In [45]:
cursor.execute("""
    DELETE FROM Customers
    WHERE Customer_ID = 1
""")

conn.commit()
#You cannot delete Customer 1 because Orders still refer to Customer 1.
cursor.execute("SELECT * FROM Customers")
print(cursor.fetchall())
print()
cursor.execute("SELECT * FROM Orders")
print(cursor.fetchall())
#ON DELETE CASCADE means deleting the parent automatically deletes the related child rows.

[(2, 'Ravi', 'ravi@gmail.com', 'Chennai'), (3, 'Priya', 'priya@gmail.com', 'Bangalore'), (4, 'Arun', 'arun@gmail.com', 'Hyderabad'), (5, 'Meena', 'meena@gmail.com', 'Pune')]

[(1002, 2, '2026-09-03', 'Shipped'), (1004, 3, '2026-09-07', 'Delivered'), (1005, 4, '2026-09-10', 'Cancelled'), (1006, 5, '2026-09-12', 'Pending')]


In [51]:
cursor.execute("""
    UPDATE Customers
    SET Customer_ID = 20
    WHERE Customer_ID = 2
""")
#If you change the parent from 2 to 20, the order would still contain 2,
#which would no longer exist in Customers.
#to remove this error we use ON UPDATE CASCADE
conn.commit()

cursor.execute("SELECT * FROM Customers")
print(cursor.fetchall())
print()
cursor.execute("SELECT * FROM Orders")
print(cursor.fetchall())
#ON UPDATE CASCADE means updating the parent automatically deletes the related child rows.

[(1, 'Anu', 'anu@gmail.com', 'Coimbatore'), (3, 'Priya', 'priya@gmail.com', 'Bangalore'), (4, 'Arun', 'arun@gmail.com', 'Hyderabad'), (5, 'Meena', 'meena@gmail.com', 'Pune'), (20, 'Ravi', 'ravi@gmail.com', 'Chennai')]

[(1001, 1, '2026-09-01', 'Delivered'), (1002, 20, '2026-09-03', 'Shipped'), (1003, 1, '2026-09-05', 'Pending'), (1004, 3, '2026-09-07', 'Delivered'), (1005, 4, '2026-09-10', 'Cancelled'), (1006, 5, '2026-09-12', 'Pending')]


In [78]:
cursor.execute("""
SELECT *
FROM(
WITH final AS(
SELECT *
FROM OrderItems AS oi
INNER JOIN Orders AS o
ON oi.OrderID = o.OrderID
INNER JOIN Products AS p
ON oi.Product_ID = p.Product_ID)

SELECT c.Customer_ID,c.CustomerName,c.Email,c.City,
        f.Product_ID, f.ProductName, f.Category,f.UnitPrice,
        f.OrderID, f.OrderDate, f.OrderStatus,
        f.Quantity, f.UnitPrice
FROM Customers AS c
LEFT JOIN final AS f
ON c.Customer_ID = f.Customer_ID) As t
WHERE t.OrderID IS NULL
""")

result = cursor.fetchall()
for r in result:
    print(r)

print()
cursor.execute("""
SELECT *
FROM(
WITH final AS(
SELECT *
FROM OrderItems AS oi
INNER JOIN Orders AS o
ON oi.OrderID = o.OrderID
INNER JOIN Products AS p
ON oi.Product_ID = p.Product_ID)

SELECT c.Customer_ID,c.CustomerName,c.Email,c.City,
        f.Product_ID, f.ProductName, f.Category,f.UnitPrice,
        f.OrderID, f.OrderDate, f.OrderStatus,
        f.Quantity, f.UnitPrice
FROM Customers AS c
LEFT JOIN final AS f
ON c.Customer_ID = f.Customer_ID) As t
WHERE t.Product_ID IS NULL OR t.OrderID IS NULL
""")

result = cursor.fetchall()
for r in result:
    print(r)

(10, 'Karthik', 'karthik@gmail.com', 'Chennai', None, None, None, None, None, None, None, None, None)

(10, 'Karthik', 'karthik@gmail.com', 'Chennai', None, None, None, None, None, None, None, None, None)


In [55]:
#perfect running of transcation
conn.execute("BEGIN TRANSACTION")

cursor.execute("""
    INSERT INTO Customers
    (Customer_ID, CustomerName, Email, City)
    VALUES (10, 'Karthik', 'karthik@gmail.com', 'Chennai')
""")

cursor.execute("""
    INSERT INTO Orders
    (OrderID, Customer_ID, OrderDate, OrderStatus)
    VALUES (1010, 10, '2026-09-23', 'Pending')
""")

conn.commit()

In [56]:
#Intentionally adding error to see the error and rollback
conn.execute("BEGIN TRANSACTION")

try:
    cursor.execute("""
        INSERT INTO Customers
        (Customer_ID, CustomerName, Email, City)
        VALUES (10, 'Karthik', 'karthik@gmail.com', 'Chennai')
    """)

    cursor.execute("""
        INSERT INTO Orders
        (OrderID, Customer_ID, OrderDate, OrderStatus)
        VALUES (1010, 10, '2026-09-23', 'Pending')
    """)
    
    cursor.execute("""
        INSERT INTO Orders
        (OrderID, Customer_ID, OrderDate, OrderStatus)
        VALUES (1011, 999, '2026-09-23', 'Pending')
    """)

    conn.commit()

except sqlite3.IntegrityError as e:
    print("Error:", e)
    conn.rollback()
    print("Transaction rolled back")

Error: UNIQUE constraint failed: Customers.Customer_ID
Transaction rolled back


In [57]:
cursor.execute("""
    CREATE TABLE IF NOT EXISTS Payments(
        PaymentID INTEGER PRIMARY KEY,
        OrderID INTEGER UNIQUE,
        PaymentDate DATE,
        Amount REAL CHECK(Amount > 0),
        PaymentStatus TEXT CHECK(
            PaymentStatus IN ('Pending', 'Successful', 'Failed')
        ),

        FOREIGN KEY(OrderID)
            REFERENCES Orders(OrderID)
    )
""")

conn.commit()

cursor.executemany("""
    INSERT INTO Payments
    (PaymentID, OrderID, PaymentDate, Amount, PaymentStatus)
    VALUES (?, ?, ?, ?, ?)
""", [
    (1, 1001, '2026-09-02', 56600, 'Successful'),
    (2, 1002, '2026-09-04', 6500, 'Successful'),
    (3, 1003, '2026-09-06', 12000, 'Pending')
])

conn.commit()

In [58]:
cursor.execute("""
    INSERT INTO Payments
    (PaymentID, OrderID, PaymentDate, Amount, PaymentStatus)
    VALUES (4, 1001, '2026-09-03', 56600, 'Successful')
""")

IntegrityError: UNIQUE constraint failed: Payments.OrderID

In [69]:
cursor.execute("""
    CREATE TABLE IF NOT EXISTS Reviews(
        CustomerID INTEGER,
        ProductID INTEGER,
        Rating INTEGER CHECK(Rating BETWEEN 1 AND 5),
        ReviewText TEXT,

        PRIMARY KEY(CustomerID, ProductID),

        FOREIGN KEY(CustomerID)
            REFERENCES Customers(Customer_ID),

        FOREIGN KEY(ProductID)
            REFERENCES Products(Product_ID)
    )
""")

conn.commit()

cursor.execute("SELECT COUNT(*) FROM Reviews")
count = cursor.fetchone()[0]
if count == 0:
    cursor.executemany("""
        INSERT INTO Reviews
        (CustomerID, ProductID, Rating, ReviewText)
        VALUES (?, ?, ?, ?)
    """, [
        (1, 101, 5, 'Excellent laptop'),
        (1, 102, 4, 'Good mouse'),
    ])

    conn.commit()
else:
    print("Already exists")

cursor.execute("SELECT * FROM Reviews")
print(cursor.fetchall())

Already exists
[(1, 101, 5, 'Excellent laptop'), (1, 102, 4, 'Good mouse')]
